# Imputation using Regression on Missing Data

## Imports

In [64]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler
import lazypredict
from sklearn.model_selection import train_test_split
import random
from sklearn import linear_model
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import *
from sklearn import tree

## Introduction of Random Deletions

In [124]:
df = pd.read_csv('data/UCI_Credit_Card.csv')

In [125]:
df

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,1,3,1,39,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,1,3,2,43,-1,-1,-1,-1,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,1,2,2,37,4,3,2,-1,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,1,3,1,41,1,-1,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


Features: ID -> PAY_AMT6

Label: default.payment.next.month

In [126]:
df.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


Randomly select the columns to replaced with missing values

In [127]:
missing_columns = ['AGE'] + [f'BILL_AMT{i}' for i in np.random.choice(a = list(range(1, 7)), size=2)]
missing_columns

['AGE', 'BILL_AMT4', 'BILL_AMT3']

Split into training and testing

In [128]:
train_df1, test_df1 = train_test_split(df, test_size=0.2)

Introduce MAR (Missing at random) values for 5% of each split

In [129]:
ds = {"train": train_df1, "test": test_df1}
for d in ds:
    mar_df = ds[d].copy(deep=True)
    actual_vals = dict()
    for col in missing_columns:
        mar_rows = ds[d][col].sample(frac = 0.05, random_state=1)
        mar_df.loc[mar_rows.index, col] = np.nan
        actual_vals[col] = ds[d][col].values
    actual_vals = pd.DataFrame(actual_vals)
    mar_df.to_csv(f'data/mar_credit_{d}.csv', index=False)
    actual_vals.to_csv(f'data/actual_vals_{d}.csv', index=False)

## Part A

Load all the six datasets

In [28]:
train_df, test_df = pd.read_csv("data/mar_credit_train.csv"), pd.read_csv("data/mar_credit_test.csv")
train_actual, test_actual = pd.read_csv("data/actual_vals_train.csv"), pd.read_csv("data/actual_vals_test.csv")
ds1 = {"train": train_df, "test": test_df}
actual = {"train": train_actual, "test": test_actual}

In [29]:
missing_columns = train_actual.columns
missing_columns

Index(['AGE', 'BILL_AMT4', 'BILL_AMT3'], dtype='object')

In [30]:
# target_missing_column = random.choice(missing_columns)
# target_missing_column
target_missing_column = 'BILL_AMT4'

In [31]:
target_classifier_feature = 'default.payment.next.month'

Categorical features:
1. ID
2. SEX
3. EDUCATION
4. MARRIAGE
5. PAY_{[0...6]}

Ordinal Categorical features:
1. AGE

Numeric features:
1. LIMIT_BAL
2. PAY_AMT{[0...6]}
3. BILL_AMT{[0...6]}

Standardizing the numeric features:

In [32]:
categorical_columns = ['ID', 'SEX', 'EDUCATION', 'MARRIAGE'] + [f'PAY_{i}' for i in range(0, 7) if i!=1]

In [33]:
numeric_columns = [
    col
    for col in ds1['train'].columns
    if col not in categorical_columns and col not in missing_columns and col != target_classifier_feature
]

In [34]:
features = categorical_columns + numeric_columns

In [35]:
for d in ds1:
    ds1[d][numeric_columns] = StandardScaler().fit_transform(ds1[d][numeric_columns])

Now, the issue is we have missing values in the columns of AGE and 2 columns of BILL_AMT (it is randomly selected while creating the dataset). We cannot use this dataset directly for the classification purpose as the missing values can hamper a classifier's performance, as they could have held key relations to other features and label necessary for making the classification. To get around this, we perform what is known as <b>Imputation</b> where we fill the missing values with some strategies.

### Imputation Method 1 : Median

We fill the missing values in a particular column with the median value of the column values. We use median over the mean as it picks the value that actually exists in the data thereby avoiding any kind of bias.

In [11]:
dc = dict()
for d in ds1:
    cleaned_df = ds1[d].copy(deep=True)
    for col in missing_columns:
        cleaned_df[col] = ds1[d][col].fillna(ds1[d][col].median())
    dc[d] = cleaned_df
    cleaned_df.to_csv(f"data/cleaned_median_{d}.csv", index=False)

### Imputation Method 2 : Linear Regression

Substituting the missing values with just the median can introduce significant bias in the features. This can negatively affect the classifier's performance. Instead, we can assume a linear relationship between the feature and rest of the features and regress over the values.

In [12]:
lin_reg_cleaned = {}
for d in ds1:
    cleaned_df = ds1[d].copy(deep=True)
    for col in ds1[d].columns:
        if not col in missing_columns:
            cleaned_df[col] = ds1[d][col]
    cleaned_df[target_missing_column] = ds1[d][target_missing_column]
    lin_reg_cleaned[d] = cleaned_df
    cleaned_df.to_csv(f"data/cleaned_linreg_{d}.csv", index=False)

In [13]:
X_train, y_train = ds1['train'][features], actual['train'][target_missing_column]
X_test, y_test = ds1['test'][features], actual['test'][target_missing_column]

In [ ]:
def linreg(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return model, mean_absolute_error(y_test, y_pred), y_pred

In [48]:
def final_df(pred, name):
    pred_df = X_test.copy(deep=True)
    pred_df[target_missing_column] = pred
    pred_df[target_classifier_feature] = test_df[target_classifier_feature]
    
    linreg_df = X_train.copy(deep=True)
    linreg_df[target_missing_column] = y_train
    linreg_df[target_classifier_feature] = train_df[target_classifier_feature]
    linreg_df = pd.concat([linreg_df, pred_df], axis=0)

    linreg_df.to_csv(f"data/{name}.csv", index=False)

    return linreg_df

#### Simple Linear Regression

In [ ]:
lr, err, pred = linreg(LinearRegression(fit_intercept=False, copy_X=True))
linreg_df = final_df(pred, "linreg_data")

#### Linear Ridge Regression

In [57]:
alphas = [0.5, 0.25]

In [58]:
for alpha in alphas:
    model, error, pred = linreg(linear_model.Ridge(alpha=alpha))
    linreg_df = final_df(pred, f"ridge_data_{alpha}")
    print(error)

6782.101219713587
6781.989669198265


#### Linear Lasso Regression

In [59]:
alphas = [0.5, 0.25]

In [60]:
for alpha in alphas:
    model, error, pred = linreg(linear_model.Lasso(alpha=alpha))
    linreg_df = final_df(pred, f"lasso_data_{alpha}")
    print(error)

6781.544661944487
6781.710818190869


### Imputation Method 3 : Non-linear Regression

Assuming a linear relationship might be too simple for the data, so we might decide to assume a more general but complex non-linear relationship between the features and then perform a non-linear regression on the same.

In [54]:
nonlin_reg_cleaned = {}
for d in ds1:
    cleaned_df = ds1[d].copy(deep=True)
    for col in ds1[d].columns:
        if not col in missing_columns:
            cleaned_df[col] = ds1[d][col]
    cleaned_df[target_missing_column] = ds1[d][target_missing_column]
    nonlin_reg_cleaned[d] = cleaned_df
    cleaned_df.to_csv(f"data/cleaned_nonlinreg_{d}.csv", index=False)

In [55]:
def nonlinreg(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return model, mean_absolute_error(y_test, y_pred), y_pred

In [56]:
def final_df(pred, name):
    pred_df = X_test.copy(deep=True)
    pred_df[target_missing_column] = pred
    pred_df[target_classifier_feature] = test_df[target_classifier_feature]
    
    nonlinreg_df = X_train.copy(deep=True)
    nonlinreg_df[target_missing_column] = y_train
    nonlinreg_df[target_classifier_feature] = train_df[target_classifier_feature]
    nonlinreg_df = pd.concat([nonlinreg_df, pred_df], axis=0)

    nonlinreg_df.to_csv(f"data/{name}.csv", index=False)

    return nonlinreg_df

#### KNeighbors Regression

In [61]:
nns = [5, 7, 10]

In [62]:
for n in nns:
    model, error, pred = nonlinreg(KNeighborsRegressor(n_neighbors=n))
    final_df(pred, f"kneighbors_{n}")
    print(error)

35577.15153333333
37558.530119047624
39564.79503333333


#### Decision Tree Regression

In [63]:
model, error, pred = nonlinreg(tree.DecisionTreeRegressor())
final_df(pred, "decisiontree")
print(error)

6540.679833333334


### Imputation Method 4 : Deletion of missing values

In [69]:
drop_df = ds1['train'].copy(deep=True)
drop_df = pd.concat([ds1['train'].dropna(), ds1['test'].dropna()])
drop_df.to_csv(f"data/dropped.csv", index=False)

## Part B

In [66]:
def report_generator(y_test, y_pred, name):
    """
        Generate a comprehensive report based on various metrics
        Implemented from sklearn
    """

    y_test = y_test.to_numpy()
    x = ["Default", "Not Default"]
    actual_default, predict_default = y_test[np.where(y_test == 1)[0]], y_pred[np.where(y_test == 1)[0]]
    actual_not_default, predict_not_default = y_test[np.where(y_test == 0)[0]], y_pred[np.where(y_test == 0)[0]]

    report = {
        "Default": {
            "precision": precision_score(actual_default, predict_default),
            "recall": recall_score(actual_default, predict_default),
            "f1": f1_score(actual_default, predict_default),
            "bal_acc": balanced_accuracy_score(actual_default, predict_default),
            "accuracy": accuracy_score(actual_default, predict_default),
            "size": len(actual_default)
        },
        "Not Default": {
            "precision": precision_score(actual_not_default, predict_not_default),
            "recall": recall_score(actual_not_default, predict_not_default),
            "f1": f1_score(actual_not_default, predict_not_default),
            "bal_acc": balanced_accuracy_score(actual_not_default, predict_not_default),
            "accuracy": accuracy_score(actual_not_default, predict_not_default),
            "size": len(actual_not_default)
        },
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "pr_curve": precision_recall_curve(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "roc_curve": roc_curve(y_test, y_pred),
        "likelihood_ratios": class_likelihood_ratios(y_test, y_pred),
        "bal_acc": balanced_accuracy_score(y_test, y_pred),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_pred),
        "avg_p": average_precision_score(y_test, y_pred)
    }
    default, not_default = report['Default'], report['Not Default']

    fig, axs = plt.subplots(3, 2, figsize= (10, 10), constrained_layout = True)
    color_palette = [
        '#0072B2',  # Cerulean Blue
        '#D55E00',  # Vermillion
    ]
    k, l = 0, 0
    for i, j in zip(default, not_default):
        if i != 'size':
            y = [default[i], not_default[j]]
            axs[k, l].bar(x, height = y, color = color_palette)
            axs[k, l].set_ylim(0, 1.1)
            axs[k, l].set_ylabel(f"{i.capitalize()} Value")
        else:
            y = [np.log(default[i]), np.log(not_default[j])]
            axs[k, l].bar(x, height = y, color = color_palette)
            axs[k, l].set_ylabel(f"Log of {i.capitalize()} Value")
        
        for q in range(len(x)):
            axs[k, l].text(q, y[q], np.round(y[q], 2), ha='center')  # Aligning text at center

        axs[k, l].set_title(f"{i.capitalize()} Comparison")
        l += 1
        if l == 2:
            k += 1
            l = 0

    fig.suptitle(f"Comparison of Metrics between Default and Not Default Classes in {name}")
    fig.align_xlabels(axs)
    plt.subplots_adjust(wspace=0.3, hspace=0.4)
    plt.savefig(f"images/{name}_report_skelearn.png")
    plt.show()

    cm = ConfusionMatrixDisplay(confusion_matrix=report['confusion_matrix'], display_labels=x[::-1])
    cm.plot()
    plt.savefig(f"images/Confusion_mat_{name}.png")
    plt.show()

    # pr = PrecisionRecallDisplay.from_predictions(y_test, y_pred)
    # pr.plot()
    # plt.savefig(f"images/PR_Curve_{name}.png")
    # plt.show()

    # roc = RocCurveDisplay.from_predictions(y_test, y_pred)
    # roc.plot()
    # plt.savefig(f"images/ROC_Curve_{name}.png")
    # plt.show()

    return report

In [ ]:
def classify(data: pd.DataFrame, name):
    train, test = train_test_split(data, test_size=0.3)
    X_train, y_train = train.drop(target_classifier_feature), train[target_classifier_feature]
    X_test, y_test = test.drop(target_classifier_feature), test[target_classifier_feature]
    
    std_scaler = StandardScaler().fit(X_train)
    X_train = std_scaler.transform(X_train)
    X_test = std_scaler.transform(X_test)

    lr = LogisticRegression()
    lr =  lr.fit(X_train, y_train)
    pred = lr.predict(X_test)

    r1 = report_generator(y_test, pred, name=name)
    
    k = ['precision','recall','f1','bal_acc','mcc','accuracy','roc_auc','avg_p']
    new_r1 = {i: r1[i] for i in k}
    new_r1['positive_likelihood_ratio'] = r1['likelihood_ratios'][0]
    new_r1['negative_likelihood_ratio'] = r1['likelihood_ratios'][1]
    new_r1['name'] = name
    df = pd.DataFrame.from_dict(new_r1, orient='index', columns=['Score']).transpose()

    return model, r1, df

In [70]:
reports, dfs = dict(), dict()

In [ ]:
data_map = {
    "median": "data/",
    "tree": "data/decisiontree.csv",
    "drop": "data/dropped.csv",
    "knn": "data/kneighbors_5.csv",
    "lasso": "data/lasso_data_0.5.csv",
    "ridge": "data/ridge_data_0.5.csv",
    "linreg": "data/linreg_data.csv"
}

In [ ]:
for d in data_map:
    print(d)
    data = pd.read_csv(data_map[d])
    model, r, report_df = classify(data, name=d)
    reports[d] = r
    dfs[d] = report_df

KeyError: "['default.payment.next.month'] not found in axis"

## Part C

### Results

In [ ]:
comp_df = pd.concat(dfs, axis=0)
comp_df.to_csv('comparison_metrics.csv', index=False)

In [ ]:
x = list(comp_df['name'])
fig, axs = plt.subplots(5, 2, figsize=(15, 15))
color_palette = [
    '#D55E00',  # Vermillion
    '#0072B2',  # Cerulean Blue
    '#009E73',  # Shamrock Green
    '#56B4E9',  # Sky Blue
    '#E69F00',  # Amber
    '#CC79A7',  # Pale Violet
    '#F0E442',  # Canaryp,  Yellow
    '#999999',   # Gray
    '#882255',
    '#1A1A1A',  # Onyx
]

i, j = 0, 0
for k in comp_df.columns:
    if k != 'name':
        ax = axs[i, j]
        l = list(comp_df[k])
        ax.bar(x, height=l, color=color_palette)
        ax.set_ylabel("Metric values")
        if i == 4:
            ax.tick_params(axis='x', labelrotation=45)
        else:
            ax.set_xticklabels([]) 
        ax.set_title(f"{k} Comparison")

        for q in range(len(l)):
            ax.text(q, l[q], np.round(l[q], 2), ha='center')  # Aligning text at center

        j += 1
        if j == 2:
            i += 1
            j = 0
fig.suptitle("Total Comparison")
plt.savefig("images/total_comp.png")
plt.show()

In [ ]:
pr_curves = {report:(reports[report]['pr_curve'], reports[report]['avg_p']) for report in reports}
roc_curves = {report:(reports[report]['roc_curve'], reports[report]['roc_auc']) for report in reports}

In [ ]:
plt.figure(figsize=(8, 6))
color_palette = [
    '#D55E00',  # Vermillion
    '#0072B2',  # Cerulean Blue
    '#009E73',  # Shamrock Green
    '#56B4E9',  # Sky Blue
    '#E69F00',  # Amber
    '#CC79A7',  # Pale Violet
    '#F0E442',  # Canary Yellow
    '#999999',   # Gray
    '#882255',
    '#1A1A1A',  # Onyx
]
k = list(roc_curves.keys())
for i in range(len(k)):
    roc, roc_auc = roc_curves[k[i]]
    fpr, tpr, thresholds = roc
    plt.plot(fpr, tpr, color=color_palette[i], lw=2, label=f"{k[i]} with auc={roc_auc:.2f}") # Plot Model 1's ROC curve

# Plot the random chance line (diagonal line)
plt.plot([0, 1], [0, 1], color=color_palette[-1], lw=2, linestyle='--', label='Random Chance')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curves for Multiple Models')
plt.legend(loc='lower right')
plt.grid(True)
plt.savefig("images/roc_curves.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
color_palette = [
    '#D55E00',  # Vermillion
    '#0072B2',  # Cerulean Blue
    '#009E73',  # Shamrock Green
    '#56B4E9',  # Sky Blue
    '#E69F00',  # Amber
    '#CC79A7',  # Pale Violet
    '#F0E442',  # Canaryp,  Yellow
    '#999999',   # Gray
    '#882255',
    '#1A1A1A',  # Onyx
]
k = list(pr_curves.keys())
for i in range(len(k)):
    pr, avg_p = pr_curves[k[i]]
    p, r, thresholds = pr
    pr_auc = auc(p, r)
    plt.plot(r, p, color=color_palette[i], lw=2, label=f"{k[i]} with auc={pr_auc:.2f}, avg_p={avg_p:.2f}") # Plot Model 1's ROC curve

# Plot the random chance line (diagonal line)
plt.plot([0, 1], [0.5, 0.5], color=color_palette[-1], lw=2, linestyle='--', label='Baseline')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves for Multiple Models')
plt.legend(loc='lower left', fontsize=8)
plt.grid(True)
plt.savefig("images/pr_curves.png")
plt.show()

### Conclusion

1. Discuss the trade-off between Listwise Deletion (Model D) and Imputation 
(Models A, B, C). Why might Model D perform poorly even if the imputed models 
perform worse? 
2. Which regression method (Linear vs. Non-Linear) performed better and why? 
Relate this to the assumed relationship between the imputed feature and the 
predictors. 
3. Conclude with a recommendation on the best strategy for handling missing data 
in this scenario, justifying your answer by referencing both the classification 
performance metrics and the conceptual implications of each method.